# Calibrating the detector's optical properties

LUCiD's forward model is **differentiable**, so we can *fit* the detector's optical
parameters (scattering, absorption, reflection, quantum efficiency, ...) to calibration
data by gradient descent — and we can compute the **best achievable** precision (the
Cramér–Rao bound, CRB) for free from the same gradients.

This notebook is a thin wrapper over three library seams in `lucid.fitting`:

| seam | role |
|---|---|
| `build_calibration_problem` | simulate truth data + set up the fit |
| `crb` | the Cramér–Rao bound (best-case 1σ per parameter) |
| `fit` | Gauss–Newton fit (with a per-PMT QE Schur block) |

We calibrate a small SK-like water Cherenkov tank from a handful of controlled light sources.

In [ ]:
import sys; sys.path.append('..')
import jax, jax.numpy as jnp, numpy as np
import matplotlib.pyplot as plt
from lucid.geometry import generate_detector
from lucid.simulation import setup_event_simulator
from lucid.sources import laser_source, isotropic_source
from lucid.detector_params import DetectorParams
from lucid.fitting import build_calibration_problem, fit, crb

GEOM = '../config/SK_like_geom_config.json'
det = generate_detector(GEOM); NS = len(det.all_points)
top, bot, R = det.H/2 - 0.1, -det.H/2 + 0.1, det.r
print(f'{NS} PMTs in this SK-like tank')

## 1. Truth detector + a diverse set of calibration sources

We pick the *true* optical parameters, then place four light sources that illuminate the
tank from complementary directions. Source **diversity** is what breaks degeneracies between
parameters (e.g. absorption vs. reflection).

In [ ]:
dp = DetectorParams.from_flat(scatter_length=70., mie_scatter_length=3000., g=0.9,
                             wall_reflection_rate=.2, sensor_reflection_rate=.2,
                             absorption_length=60., qe=0.07, qe_corrections=jnp.ones(NS))

sources = [laser_source(position=[0, 0, top], direction=[0, 0, -1], intensity=1e6),   # downward
           laser_source(position=[0, 0, bot], direction=[0, 0,  1], intensity=1e6),   # upward
           laser_source(position=[R-.1, 0, 0], direction=[-1, 0, 0], intensity=1e6),  # from the wall
           isotropic_source(position=[0, 0, 0], intensity=1e6)]                        # central flasher

sim = setup_event_simulator(GEOM, 1_000_000, temperature=None, K=8, is_calibration=True,
                            hit_mode='aggregated', wavelength_mode=False,
                            n_cap=100, n_angular=150, n_height=100)
print('calibration simulator ready')

## 2. The Cramér–Rao bound — how well *can* we do?

`build_calibration_problem` simulates the truth charge for each source and packages the
fit. `crb` then returns the fractional 1σ lower bound on each parameter, straight from the
Fisher information at truth.

In [ ]:
FIELDS = ['g', 'scatter_length', 'mie_scatter_length', 'absorption_length',
          'wall_reflection_rate', 'sensor_reflection_rate', 'qe']
prob = build_calibration_problem(sim, sources, dp, FIELDS, key=jax.random.PRNGKey(1))
sigma = crb(prob['source_models'], prob['theta_true'], NS)['sigma']

print('CRB (fractional 1σ) per parameter:')
for f, s in zip(FIELDS, sigma):
    print(f'  {f:22s} {s:6.2%}')

## 3. Fit from a perturbed start

We start the parameters 15% away from truth and run the Gauss–Newton `fit`. The per-PMT QE
(~thousands of nuisance parameters) is marginalised analytically via a Schur complement, so
the fit solves only for the global optical parameters.

In [ ]:
start = prob['theta0'] + np.random.default_rng(0).uniform(-.15, .15, prob['theta0'].shape)
res = fit(prob['source_models'], prob['truth_charge'], start, NS, steps=300, refresh=15, nb_h=2)

truth = np.exp(prob['theta0'])
print(f'{"param":22s}{"truth":>9s}{"start":>9s}{"fit":>9s}{"err":>8s}{"CRB":>7s}')
errs = []
for i, f in enumerate(FIELDS):
    e = res['theta'][i] / truth[i] - 1; errs.append(e)
    print(f'{f:22s}{truth[i]:9.3f}{np.exp(start[i]):9.3f}{res["theta"][i]:9.3f}{e:+7.1%}{sigma[i]:7.1%}')
errs = np.array(errs)

## 4. Fit error vs. the Cramér–Rao bound

A well-behaved fit lands at or near the CRB — the parameters it recovers tightly are the
ones the data actually constrains.

In [ ]:
x = np.arange(len(FIELDS))
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - .2, np.abs(errs) * 100, .4, label='|fit error|', color='#3477b8')
ax.bar(x + .2, np.array(sigma) * 100, .4, label='CRB (1σ)', color='#c0392b', alpha=.7)
ax.set_xticks(x); ax.set_xticklabels(FIELDS, rotation=35, ha='right')
ax.set_ylabel('%'); ax.set_title('Calibration: fit error vs Cramér–Rao bound')
ax.legend(); ax.grid(alpha=.3, axis='y'); fig.tight_layout(); plt.show()

## Takeaways

- The detector's optical parameters are recovered by **gradient-descent calibration** on
  controlled light sources — no hand-tuning.
- The **CRB** tells you, before fitting, which parameters the data constrains; the fit error
  tracking the CRB is the signature of a healthy calibration.
- The same `build_calibration_problem` / `fit` / `crb` seams extend to **per-PMT QE**,
  **wavelength-dependent** optics, and **scintillation** parameters (see the calibration
  gradient and scintillation notebooks).